# P4-7 · MCP và hệ sinh thái tool

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

In [ ]:
!pip install -q "mcp<2"

## Lý thuyết

In [ ]:
from mcp.server.fastmcp import FastMCP
mcp = FastMCP("vi-du")

@mcp.tool()
def orders(q: str) -> str:
    """Orders."""

In [ ]:
from typing import Literal

@mcp.tool()
def search_orders(phone: str, status: Literal["đang giao", "đã giao", "đã hủy"] | None = None,
                  limit: int = 5) -> list[dict]:
    """Tìm đơn của một khách theo số điện thoại (10 chữ số).
    Lọc theo trạng thái nếu cần. Trả về tối đa `limit` đơn, mới nhất trước.
    Dùng khi khách không nhớ mã đơn. Nếu đã có mã đơn, dùng get_order."""

# model chỉ thấy tên, mô tả và schema tham số của tool (await chạy được trong notebook)
for t in await mcp.list_tools():
    print(t.name, "|", t.description.splitlines()[0], "|", t.inputSchema["properties"])

orders | Orders. | {'q': {'title': 'Q', 'type': 'string'}}
search_orders | Tìm đơn của một khách theo số điện thoại (10 chữ số). | {'phone': {'title': 'Phone', 'type': 'string'}, 'status': {'anyOf': [{'enum': ['đang giao', 'đã giao', 'đã hủy'], 'type': 'string'}, {'type': 'null'}], 'default': None, 'title': 'Status'}, 'limit': {'default': 5, 'title': 'Limit', 'type': 'integer'}}


In [ ]:
%%writefile server.py
from typing import Literal
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("shop-orders")
ORDERS = {"DH1024": {"status": "đang giao", "eta": "2026-11-06", "items": ["bàn trang điểm"], "phone": "0901234567"},
          "DH1025": {"status": "đã giao", "eta": "2026-11-02", "items": ["ghế"], "phone": "0901234567"}}

@mcp.tool()
def get_order(order_id: str) -> dict:
    """Tra một đơn hàng theo mã đơn (dạng DH + 4 chữ số, ví dụ DH1024).
    Trả về trạng thái, ngày giao dự kiến và sản phẩm. Dùng khi khách hỏi đơn của họ."""
    if order_id not in ORDERS:
        return {"error": f"Không có đơn {order_id}. Kiểm tra lại mã, dạng DH + 4 chữ số."}
    o = ORDERS[order_id]
    return {k: o[k] for k in ("status", "eta", "items")}        # không trả số điện thoại ra ngoài

@mcp.tool()
def search_orders(phone: str, status: Literal["đang giao", "đã giao", "đã hủy"] | None = None,
                  limit: int = 5) -> list[dict]:
    """Tìm đơn của một khách theo số điện thoại (10 chữ số). Lọc theo trạng thái nếu cần."""
    rows = [{"order_id": k, "status": v["status"]} for k, v in ORDERS.items()
            if v["phone"] == phone and (status is None or v["status"] == status)]
    return rows[:limit]

@mcp.resource("orders://summary")
def orders_summary() -> str:
    """Tổng quan số đơn theo trạng thái."""
    counts = {}
    for o in ORDERS.values():
        counts[o["status"]] = counts.get(o["status"], 0) + 1
    return "\n".join(f"{k}: {v}" for k, v in counts.items())

@mcp.prompt()
def weekly_report(week: str) -> str:
    """Mẫu prompt báo cáo đơn hàng theo tuần."""
    return f"Tóm tắt tình hình đơn hàng tuần {week}: số đơn mới, đơn giao trễ, khiếu nại."

if __name__ == "__main__":
    mcp.run()          # mặc định chạy qua stdio

Writing server.py


In [ ]:
%%writefile client.py
import asyncio, os, sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def main():
    params = StdioServerParameters(command=sys.executable, args=["server.py"], env=dict(os.environ))
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            print([t.name for t in tools.tools])                        # ['get_order', 'search_orders']
            r = await session.call_tool("get_order", {"order_id": "DH9"})
            print(r.content[0].text)    # {"error": "Không có đơn DH9. Kiểm tra lại mã, dạng DH + 4 chữ số."}
            res = await session.read_resource("orders://summary")
            print(res.contents[0].text)                                 # đang giao: 1 / đã giao: 1

asyncio.run(main())

Writing client.py


In [ ]:
!python client.py

['get_order', 'search_orders']
{
  "error": "Không có đơn DH9. Kiểm tra lại mã, dạng DH + 4 chữ số."
}
đang giao: 1
đã giao: 1


## Bài tập 5: MCP client {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
%%writefile check_server.py
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def main():
    params = StdioServerParameters(command="python", args=["server.py"])
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            print([t.name for t in tools.tools])
            result = await session.call_tool("get_order", {"order_id": "DH1024"})
            print(result.content)

asyncio.run(main())

Writing check_server.py


In [ ]:
!python check_server.py

['get_order', 'search_orders']
[TextContent(type='text', text='{\n  "status": "đang giao",\n  "eta": "2026-11-06",\n  "items": [\n    "bàn trang điểm"\n  ]\n}', annotations=None, meta=None)]
